# IBM HR Employee Attrition Analytics

This notebook implements the supplied submission roadmap using the IBM HR Employee Attrition dataset from Kaggle. The original roadmap is written for e-commerce transactions, so this version uses employee-level workforce features instead of inventing transaction/RFM fields.

**Guardrail:** model probabilities are directional prioritization aids for human review, not causal explanations or employment decisions.

## 1. Setup and data acquisition

The CSV is included in `data/` for reproducibility. The source URL is the Kaggle page linked in `README.md`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from run_pipeline import clean_data, save_eda_figures, train_model, write_summary, DATA_PATH, OUTPUT_DIR

df = clean_data(DATA_PATH)
df.head()

## 2. Data quality and dictionary

The first header in the Kaggle export contains a UTF-8 BOM. The cleaning function removes it, trims string fields, handles future missingness explicitly, and creates a numeric target flag.

In [ ]:
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Missing cells:', int(df.isna().sum().sum()))
print('Observed attrition:', df['Attrition'].value_counts().to_dict())
display(df.dtypes.to_frame('dtype').head(20))

## 3. EDA using Observation → Insight → Hypothesis → Recommendation

The five exported figures cover tenure, department, overtime, role, and overtime-by-attrition context. The Word report contains the full four-step interpretation for each.

In [ ]:
save_eda_figures(df)
display(df.groupby('OverTime')['AttritionFlag'].mean().mul(100).round(2).rename('attrition_rate_pct'))
display(df.groupby('JobRole')['AttritionFlag'].agg(['count', 'mean']).assign(attrition_rate_pct=lambda x: x['mean'] * 100).sort_values('attrition_rate_pct', ascending=False))

## 4. Leakage-aware Logistic Regression

The target is `Attrition == 'Yes'`. Inputs are limited to employee context, operating patterns, satisfaction, tenure, mobility, and role features. Preprocessing is fit only within the training pipeline, and the test set is stratified and held out.

In [ ]:
model, metrics, risk_scores = train_model(df)
print(json.dumps({k: metrics[k] for k in ['accuracy', 'precision', 'recall', 'roc_auc', 'confusion_matrix']}, indent=2))
risk_scores.head(10)

## 5. Risk tiers and outputs

The supplied thresholds are applied to predicted probability: Low `< 40%`, Medium `40–70%`, and High `> 70%`. Exported files include cleaned data, model metrics, employee risk scores, and five figures.

In [ ]:
write_summary(df, metrics, risk_scores)
print(json.dumps(json.loads((OUTPUT_DIR / 'analysis_summary.json').read_text()), indent=2)[:2500])

## 6. Business recommendations

1. Audit overtime concentration by role, team, and manager before applying broad incentives.
2. Strengthen early-tenure onboarding with 30/60/90-day check-ins and role-clarity prompts.
3. Use role and department comparisons for qualitative listening, not for labeling individuals.
4. Pair satisfaction signals with manager coaching and work-life-balance interventions.
5. Track calibration, fairness, and intervention outcomes over time; do not optimize only for accuracy.